# 02 - Pipeline complet : analyse, features, modèles additifs

Notebook autonome pour **Google Colab**. Il reproduit le pipeline de `src/`
(`features.py`, `build_dataset.py`, `train.py`, `predict.py`) pour le défi Zindi
*Financial Stress Prediction Challenge (September Edition)*. Le journal complet
du raisonnement, avec chaque expérience et son score, est dans `experiments.md`.

**Métrique** (formule retrouvée à partir de deux scores publics connus) :
`score = 0.4 × AUC + 0.6 × (1 − logloss / 0.595)`. Un millième de logloss vaut
donc un millième de score, contre 0.4 millième pour un millième d'AUC.

**Démarche en bref**
1. Point de départ : le starter notebook Zindi (régression logistique, AUC 0.753).
2. Analyse : chaque client a 7 instantanés, et la cible se lit dans la rupture entre
   les 3 mois récents et les 3 mois anciens.
3. Rétro-ingénierie : les données sont stationnaires et la cible est calculée à partir
   des features (un score + du bruit). Ce score est une **somme** de quelques facteurs
   (ratio sorties/entrées, baisse des entrées, baisse du solde, apparition de retraits
   ou de paiements, volatilité du solde, activité). Volumes, montants max et
   contreparties sont du bruit.
4. Le bruit ajouté au score avant de retenir les 15 % de lignes les plus risquées est
   **gaussien** : P(stress | x) = Φ(score). Les modèles sont donc additifs sur l'échelle
   **probit** (un modèle additif en logit surestime les faibles risques).
5. Modèles : trois modèles **additifs** sur les 15 features les mieux classées —
   LightGBM probit à arbres d'une division (contraintes de monotonie), GAM probit à
   P-splines (mêmes contraintes) et CatBoost à arbres d'une division — validation
   croisée stratifiée à 10 plis × 3 seeds, puis blend sur l'échelle probit.

Seed fixée partout (`SEED = 42`), comme l'exige le règlement.

## 1. Installation et configuration

In [ ]:
!pip install -q lightgbm catboost

In [ ]:
import itertools
import os

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy import optimize, special
from scipy.interpolate import BSpline
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_FOLDS = 10
BAGGING_SEEDS = [42, 123, 2024]
# Pour un essai rapide, réduire N_FOLDS à 5 et BAGGING_SEEDS à [42] : le score obtenu
# sera alors légèrement différent de celui de la soumission.

## 2. Chargement des données

Dépose `Train.csv` et `Test.csv` (page *Data* du défi sur zindi.world). La cellule
cherche d'abord `data/raw/`, puis propose l'upload Colab.

In [ ]:
RAW_DIR = "data/raw" if os.path.exists("data/raw") else "../data/raw"
if os.path.exists(f"{RAW_DIR}/Train.csv") and os.path.exists(f"{RAW_DIR}/Test.csv"):
    train_path, test_path = f"{RAW_DIR}/Train.csv", f"{RAW_DIR}/Test.csv"
else:
    from google.colab import files
    print("Upload de Train.csv et Test.csv :")
    files.upload()
    train_path, test_path = "Train.csv", "Test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
TARGET = "liquidity_stress_next_30d"
print(train.shape, test.shape, "taux de stress :", train[TARGET].mean())

## 3. Analyse : ce qui a guidé le feature engineering

Cinq constats, chacun vérifié dans `experiments.md`.

**3.1 Structure client.** En regroupant les lignes par profil (ARPU, âge,
région, segment…), on trouve 10 000 clients ayant chacun 7 instantanés : 4 en
train, 3 en test. Les 42 mois d'un client sont tous distincts, il n'y a donc
**aucune fuite temporelle** entre instantanés. La cible est quasi indépendante
d'un instantané à l'autre, et aucune feature n'utilise les données du test.

In [ ]:
PROFILE = ["arpu", "age", "gender", "region", "smartphone", "segment", "earning_pattern", "x_90_d_activity_rate"]
both = pd.concat([train.assign(_src="train"), test.assign(_src="test")], ignore_index=True)
both["client"] = both.groupby(PROFILE, sort=False).ngroup()
print("clients :", both["client"].nunique())
print("instantanés par client (train + test) :", both.groupby("client").size().value_counts().to_dict())
print("taux de stress par position dans le bloc client (train) :",
      both[both._src == "train"].groupby(both.groupby(["_src", "client"]).cumcount())[TARGET].mean().round(4).to_dict())

**3.2 La cible se lit par blocs de 3 mois.** Chez les instantanés stressés,
l'écart avec les non-stressés bascule entre M4 et M3, et reste plat à l'intérieur
de chaque bloc. Dans le bloc récent, les entrées chutent et les sorties montent.

In [ ]:
y_tr = train[TARGET]
rows = {}
for serie in ["daily_avg_bal", "deposit_total_value", "received_total_value", "withdraw_total_value", "paybill_total_value"]:
    lv = np.log1p(train[[f"m{m}_{serie}" for m in range(6, 0, -1)]])
    rows[serie] = (lv[y_tr == 1].mean() - lv[y_tr == 0].mean()).to_numpy().round(2)
pd.DataFrame(rows, index=["M6", "M5", "M4", "M3", "M2", "M1"]).T

**3.3 Additionner les flux par famille fait ressortir le signal.** Le ratio
entrées récentes / anciennes (dépôts + reçus + virements bancaires) prédit bien
mieux que chaque type pris seul. Quand les entrées tombent sous 5 % de leur niveau
ancien, environ 95 % des instantanés sont stressés.

In [ ]:
inflow = sum(train[[f"m{m}_{t}_total_value" for m in range(1, 7)]].to_numpy(dtype=float)
             for t in ["deposit", "received", "transfer_from_bank"])
ratio = (inflow[:, :3].mean(1) + 1) / (inflow[:, 3:].mean(1) + 1)
print("AUC univarié du ratio des entrées :", round(1 - roc_auc_score(y_tr, ratio), 4))
pd.Series(y_tr.to_numpy()).groupby(pd.cut(ratio, [0, 0.05, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 1e9])).agg(["mean", "size"]).round(3)

**3.4 La cible est calculée à partir des features.** Sur l'ensemble des lignes,
bloc récent et bloc ancien ont la même distribution (le générateur produit des
données stationnaires), mais pas chez les non-stressés seuls : le générateur ne
modifie pas les stressés, il étiquette les lignes dont un score (fonction des
features + bruit) est le plus élevé. Seule l'information utilisée par ce score
compte, ce qui a guidé la recherche de facteurs.

In [ ]:
rows = {}
for t in ["deposit", "received", "withdraw"]:
    act = train[[f"m{m}_{t}_total_value" for m in range(1, 7)]].to_numpy() > 0
    rows[t] = {"toutes lignes : récent": act[:, :3].mean(), "toutes lignes : ancien": act[:, 3:].mean(),
               "non stressés : récent": act[y_tr == 0, :3].mean(), "non stressés : ancien": act[y_tr == 0, 3:].mean()}
pd.DataFrame(rows).T.round(3)

**3.5 Paybill et paiements marchands forment une seule catégorie.** Une
apparition de paybill (absent des 3 mois anciens, présent dans les 3 récents) ne
change rien si les paiements marchands étaient déjà actifs ; des paiements qui
apparaissent alors qu'aucun des deux n'existait triplent presque le taux de stress.

In [ ]:
def etat(a):
    rec, old = a[:, :3].sum(1), a[:, 3:].sum(1)
    return np.select([(old == 0) & (rec > 0), (old > 0) & (rec == 0), (old == 0) & (rec == 0)],
                     ["apparu", "arrêté", "jamais"], "actif")


pb = train[[f"m{m}_paybill_total_value" for m in range(1, 7)]].to_numpy()
mp = train[[f"m{m}_merchantpay_total_value" for m in range(1, 7)]].to_numpy()
pd.crosstab(etat(pb), etat(mp), values=y_tr, aggfunc="mean").round(3).rename_axis(index="paybill", columns="marchands")

## 4. Feature engineering

Code identique à `src/features.py`.

In [ ]:
import numpy as np
import pandas as pd

MONTHS = [1, 2, 3, 4, 5, 6]

# Seule métrique utilisée par type de transaction (voir docstring du module).
TRANSACTION_TYPES = {
    "paybill": ["total_value"],
    "merchantpay": ["total_value"],
    "transfer_from_bank": ["total_value"],
    "mm_send": ["total_value"],
    "received": ["total_value"],
    "deposit": ["total_value"],
    "withdraw": ["total_value"],
}

OUTFLOW_TYPES = ["paybill", "merchantpay", "mm_send", "withdraw"]
INFLOW_TYPES = ["deposit", "received", "transfer_from_bank"]

EPS = 1e-6


def _monthly_columns(prefix_fn):
    return [prefix_fn(m) for m in MONTHS]


def _series_features(df: pd.DataFrame, cols: list[str], name: str) -> pd.DataFrame:
    """Niveau, volatilité et momentum (bloc récent vs bloc ancien) d'une série de 6 mois.

    Seuls les totaux par bloc de 3 mois et la volatilité sur 6 mois portent du signal
    (experiments.md, 2026-10-10) : la tendance, le dernier mois seul, le ratio M1/M2
    et les min/max par bloc ont été retirés sans perte (LightGBM 0.23146 sans eux
    contre 0.23155 avec).
    """
    values = df[cols].to_numpy(dtype=float)
    recent = values[:, :3]  # M1, M2, M3
    older = values[:, 3:]  # M4, M5, M6

    mean = values.mean(axis=1)
    std = values.std(axis=1)

    out = pd.DataFrame(index=df.index)
    out[f"{name}_sum_6m"] = values.sum(axis=1)
    out[f"{name}_mean_6m"] = mean
    out[f"{name}_std_6m"] = std
    out[f"{name}_cv_6m"] = std / (np.abs(mean) + EPS)
    out[f"{name}_momentum_recent_vs_old"] = (recent.mean(axis=1) + EPS) / (
        older.mean(axis=1) + EPS
    )
    return out


def build_transaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features de niveau/volatilité/momentum pour chaque (type de transaction, métrique)."""
    blocks = []
    for ttype, metrics in TRANSACTION_TYPES.items():
        for metric in metrics:
            cols = _monthly_columns(lambda m, t=ttype, me=metric: f"m{m}_{t}_{me}")
            blocks.append(_series_features(df, cols, f"{ttype}_{metric}"))
    return pd.concat(blocks, axis=1)


def build_balance_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features de niveau/volatilité/momentum pour le solde journalier moyen."""
    cols = _monthly_columns(lambda m: f"m{m}_daily_avg_bal")
    return _series_features(df, cols, "daily_avg_bal")


def _type_total_value_6m(df: pd.DataFrame, ttype: str) -> pd.Series:
    cols = _monthly_columns(lambda m, t=ttype: f"m{m}_{t}_total_value")
    return df[cols].sum(axis=1)


def build_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ratios inter-types de transaction (flux sortants/entrants, retraits/dépôts)."""
    out = pd.DataFrame(index=df.index)

    outflow_value = sum(_type_total_value_6m(df, t) for t in OUTFLOW_TYPES)
    inflow_value = sum(_type_total_value_6m(df, t) for t in INFLOW_TYPES)
    out["outflow_to_inflow_value_ratio"] = (outflow_value + EPS) / (inflow_value + EPS)

    withdraw_value = _type_total_value_6m(df, "withdraw")
    deposit_value = _type_total_value_6m(df, "deposit")
    out["withdraw_to_deposit_value_ratio"] = (withdraw_value + EPS) / (
        deposit_value + EPS
    )
    return out


ALL_TYPES = list(TRANSACTION_TYPES.keys())


def build_overall_activity_features(df: pd.DataFrame) -> pd.DataFrame:
    """Niveau/volatilité/momentum de l'activité tous types confondus : un client peut
    basculer d'un type à l'autre (paybill -> mm_send) sans que son activité globale
    ne bouge, signal invisible aux features par type."""
    value_cols = [
        _monthly_columns(lambda m, t=ttype: f"m{m}_{ttype}_total_value")
        for ttype in ALL_TYPES
    ]
    total_value_per_month = sum(df[cols].to_numpy(dtype=float) for cols in value_cols)
    tmp_value = pd.DataFrame(
        total_value_per_month, columns=_monthly_columns(lambda m: f"m{m}"), index=df.index
    )
    return _series_features(tmp_value, tmp_value.columns.tolist(), "overall_value")


def build_share_features(df: pd.DataFrame) -> pd.DataFrame:
    """Part de chaque type dans le total des sorties/entrées sur 6 mois : capture la
    composition du comportement (cash vs factures), pas seulement son niveau."""
    out = pd.DataFrame(index=df.index)

    outflow_value = sum(_type_total_value_6m(df, t) for t in OUTFLOW_TYPES)
    for t in OUTFLOW_TYPES:
        out[f"{t}_share_of_outflow"] = (_type_total_value_6m(df, t) + EPS) / (
            outflow_value + EPS
        )

    inflow_value = sum(_type_total_value_6m(df, t) for t in INFLOW_TYPES)
    for t in INFLOW_TYPES:
        out[f"{t}_share_of_inflow"] = (_type_total_value_6m(df, t) + EPS) / (
            inflow_value + EPS
        )

    return out


def build_balance_to_income_features(df: pd.DataFrame) -> pd.DataFrame:
    """Solde rapporté au revenu (ARPU) : un solde bas n'inquiète pas pareil selon
    le revenu du client, ce qu'aucune des deux variables seule ne capture."""
    bal_cols = _monthly_columns(lambda m: f"m{m}_daily_avg_bal")
    bal_mean = df[bal_cols].mean(axis=1)
    out = pd.DataFrame(index=df.index)
    out["balance_to_arpu_ratio"] = bal_mean / (df["arpu"] + EPS)
    return out


FLOW_SMOOTHING = 1.0  # lissage additif des ratios : évite qu'un mois quasi nul fasse exploser un ratio


def _block_features(values: np.ndarray, name: str, out: dict) -> None:
    """Compare le bloc récent (M1-M3) au bloc ancien (M4-M6) d'une série mensuelle.

    L'analyse (experiments.md, 2026-10-10) montre que la cible se lit au niveau des
    blocs de 3 mois : chez les instantanés stressés, l'écart avec les non-stressés
    est quasi constant à l'intérieur de chaque bloc et bascule entre M4 et M3.
    """
    rec, old = values[:, :3], values[:, 3:]
    out[f"{name}_rec_mean"] = rec.mean(axis=1)
    out[f"{name}_old_mean"] = old.mean(axis=1)
    out[f"{name}_ratio"] = (rec.mean(axis=1) + FLOW_SMOOTHING) / (old.mean(axis=1) + FLOW_SMOOTHING)
    out[f"{name}_logratio"] = np.log1p(rec).mean(axis=1) - np.log1p(old).mean(axis=1)


def _monthly_array(df: pd.DataFrame, name: str) -> np.ndarray:
    return df[_monthly_columns(lambda m: f"m{m}_{name}")].to_numpy(dtype=float)


def build_flow_aggregate_features(df: pd.DataFrame) -> pd.DataFrame:
    """Agrégats des flux entrants / sortants et événements d'arrêt / apparition.

    Un type pris seul est bruité ; sommer entrées (dépôts + reçus + virements) et
    sorties (retraits + paybill + paiements marchands + envois) fait ressortir le
    signal (AUC univarié du ratio récent/ancien : 0.72, contre 0.66 pour le meilleur
    type isolé ; entrées sous 5% du niveau ancien -> 95% de stress).

    Les indicateurs d'arrêt / apparition captent ce que les ratios lissent : dépôts
    arrêtés -> 37% de stress, retraits qui apparaissent -> 44%.
    """
    out = {}
    inflow_value = sum(_monthly_array(df, f"{t}_total_value") for t in INFLOW_TYPES)
    outflow_value = sum(_monthly_array(df, f"{t}_total_value") for t in OUTFLOW_TYPES)
    balance = _monthly_array(df, "daily_avg_bal")

    _block_features(inflow_value, "inflow_value", out)
    _block_features(outflow_value, "outflow_value", out)
    _block_features(balance, "bal", out)

    net = inflow_value - outflow_value
    out["net_rec_sum"] = net[:, :3].sum(axis=1)
    out["net_old_sum"] = net[:, 3:].sum(axis=1)
    out["net_rec_over_bal_old"] = net[:, :3].sum(axis=1) / (balance[:, 3:].mean(axis=1) + FLOW_SMOOTHING)
    out["net_rec_over_inflow_old"] = net[:, :3].sum(axis=1) / (inflow_value[:, 3:].sum(axis=1) + FLOW_SMOOTHING)
    out["outflow_over_inflow_rec"] = (outflow_value[:, :3].sum(axis=1) + FLOW_SMOOTHING) / (
        inflow_value[:, :3].sum(axis=1) + FLOW_SMOOTHING
    )
    out["outflow_over_inflow_old"] = (outflow_value[:, 3:].sum(axis=1) + FLOW_SMOOTHING) / (
        inflow_value[:, 3:].sum(axis=1) + FLOW_SMOOTHING
    )

    n_stopped = np.zeros(len(df))
    n_started = np.zeros(len(df))
    for t in TRANSACTION_TYPES:
        value = _monthly_array(df, f"{t}_total_value")
        rec_zero = (value[:, :3] == 0).all(axis=1)
        old_zero = (value[:, 3:] == 0).all(axis=1)
        stopped = (rec_zero & ~old_zero).astype(float)
        started = (~rec_zero & old_zero).astype(float)
        out[f"{t}_stopped"] = stopped
        out[f"{t}_started"] = started
        out[f"{t}_rec_zero_months"] = (value[:, :3] == 0).sum(axis=1)
        out[f"{t}_old_zero_months"] = (value[:, 3:] == 0).sum(axis=1)
        n_stopped += stopped
        n_started += started
    out["n_types_stopped"] = n_stopped
    out["n_types_started"] = n_started
    out["inflow_types_stopped"] = sum(out[f"{t}_stopped"] for t in INFLOW_TYPES)
    out["outflow_types_started"] = sum(out[f"{t}_started"] for t in OUTFLOW_TYPES)
    return pd.DataFrame(out, index=df.index)


# Paybill et paiements marchands se comportent comme une seule catégorie "paiements" :
# une apparition de paybill quand les paiements marchands sont déjà actifs laisse le
# stress au taux de base (14%), contre 34% quand les deux apparaissent depuis rien
# (40% si ensemble). Ajouter retraits ou envois à ce groupe n'apporte rien (testé).
CATEGORY_GROUPS = {
    "payments": ["paybill", "merchantpay"],
}


def build_category_features(df: pd.DataFrame) -> pd.DataFrame:
    """Comparaisons bloc récent / bloc ancien par catégorie regroupée, et ratios lissés.

    - Par catégorie (CATEGORY_GROUPS) : comparaisons de blocs (_block_features),
      apparition/arrêt, mois actifs par bloc.
    - Par type : ratio des moyennes de bloc lissé à 1 unité monétaire (FLOW_SMOOTHING),
      contre 1e-6 pour les ratios `_momentum_recent_vs_old` — avec un lissage de 1, une
      apparition vaut son montant récent, comparable à une hausse de même ampleur
      (AUC 0.82 pour le montant des retraits qui apparaissent).
    - Paiements et retraits rapportés aux entrées de chaque bloc, sur le même principe
      que `outflow_over_inflow_rec`, la feature la plus importante du modèle.
    - Arrêt simultané des dépôts et réceptions (79% de stress, contre 37% et 28% pour
      chacun pris seul).
    """
    out = {}
    for name, types in CATEGORY_GROUPS.items():
        values = sum(_monthly_array(df, f"{t}_total_value") for t in types)
        _block_features(values, f"{name}_value", out)
        rec_sum, old_sum = values[:, :3].sum(axis=1), values[:, 3:].sum(axis=1)
        out[f"{name}_started"] = ((old_sum == 0) & (rec_sum > 0)).astype(float)
        out[f"{name}_stopped"] = ((rec_sum == 0) & (old_sum > 0)).astype(float)
        out[f"{name}_rec_active_months"] = (values[:, :3] > 0).sum(axis=1)
        out[f"{name}_old_active_months"] = (values[:, 3:] > 0).sum(axis=1)

    for t in TRANSACTION_TYPES:
        values = _monthly_array(df, f"{t}_total_value")
        out[f"{t}_block_ratio"] = (values[:, :3].mean(axis=1) + FLOW_SMOOTHING) / (
            values[:, 3:].mean(axis=1) + FLOW_SMOOTHING
        )

    inflow = sum(_monthly_array(df, f"{t}_total_value") for t in INFLOW_TYPES)
    payments = sum(_monthly_array(df, f"{t}_total_value") for t in CATEGORY_GROUPS["payments"])
    withdraw = _monthly_array(df, "withdraw_total_value")
    for block, cols in [("rec", slice(0, 3)), ("old", slice(3, 6))]:
        inflow_sum = inflow[:, cols].sum(axis=1) + FLOW_SMOOTHING
        out[f"payments_over_inflow_{block}"] = (payments[:, cols].sum(axis=1) + FLOW_SMOOTHING) / inflow_sum
        out[f"withdraw_over_inflow_{block}"] = (withdraw[:, cols].sum(axis=1) + FLOW_SMOOTHING) / inflow_sum

    dep_rec = _monthly_array(df, "deposit_total_value") + _monthly_array(df, "received_total_value")
    out["deposit_received_stopped"] = (
        (dep_rec[:, :3].sum(axis=1) == 0) & (dep_rec[:, 3:].sum(axis=1) > 0)
    ).astype(float)
    return pd.DataFrame(out, index=df.index)


PASSTHROUGH_COLUMNS = [
    "arpu",
    "age",
    "x_90_d_activity_rate",
    "gender",
    "region",
    "smartphone",
    "segment",
    "earning_pattern",
]


GROUP_RELATIVE_COLS = [
    "daily_avg_bal_mean_6m",
    "outflow_to_inflow_value_ratio",
    "balance_to_arpu_ratio",
]


def fit_group_stats(features_df: pd.DataFrame, group_col: str = "segment", cols=None):
    """Calcule moyenne/écart-type par groupe (segment) sur le train UNIQUEMENT,
    comme un scaler : fit sur train, puis appliqué tel quel au test (apply_group_relative_features)
    pour ne jamais faire dépendre une feature de train de la distribution du test."""
    cols = cols or GROUP_RELATIVE_COLS
    return features_df.groupby(group_col)[cols].agg(["mean", "std"])


def apply_group_relative_features(
    features_df: pd.DataFrame, group_stats, group_col: str = "segment", cols=None
) -> pd.DataFrame:
    """Z-score de quelques features clés par rapport à leur groupe (segment) : un
    solde ou un ratio bas n'a pas le même sens pour un client HVC que pour un LVC."""
    cols = cols or GROUP_RELATIVE_COLS
    out = pd.DataFrame(index=features_df.index)
    for col in cols:
        means = features_df[group_col].map(group_stats[(col, "mean")])
        stds = features_df[group_col].map(group_stats[(col, "std")])
        z = (features_df[col] - means) / (stds.replace(0, np.nan) + EPS)
        out[f"{col}_z_by_segment"] = z.fillna(0.0)
    return out


def build_segment_cross_feature(df: pd.DataFrame) -> pd.DataFrame:
    """Croisement segment x earning_pattern : les arbres ne retrouvent une
    interaction entre deux catégorielles que si la profondeur le permet. On la
    fournit directement pour ne pas dépendre de la profondeur choisie."""
    out = pd.DataFrame(index=df.index)
    out["segment_x_earning_pattern"] = (
        df["segment"].astype(str) + "_" + df["earning_pattern"].astype(str)
    )
    return out


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Construit le dataset de features par client à partir des données brutes Zindi."""
    blocks = [
        df[PASSTHROUGH_COLUMNS],
        build_transaction_features(df),
        build_balance_features(df),
        build_ratio_features(df),
        build_overall_activity_features(df),
        build_share_features(df),
        build_balance_to_income_features(df),
        build_segment_cross_feature(df),
        build_flow_aggregate_features(df),
        build_category_features(df),
    ]
    features = pd.concat(blocks, axis=1)
    features.index = df["ID"] if "ID" in df.columns else df.index
    return features

In [ ]:
feats_train = build_features(train)
feats_test = build_features(test)

# Z-scores par segment : statistiques ajustées sur le train uniquement (comme un
# scaler), puis appliquées telles quelles au test.
group_stats = fit_group_stats(feats_train)
feats_train = pd.concat([feats_train, apply_group_relative_features(feats_train, group_stats)], axis=1)
feats_test = pd.concat([feats_test, apply_group_relative_features(feats_test, group_stats)], axis=1)

y = train[TARGET].to_numpy()
cat_cols = [c for c in feats_train.columns if feats_train[c].dtype == object or str(feats_train[c].dtype) in ("str", "string")]
for c in cat_cols:
    feats_train[c] = feats_train[c].astype("category")
    feats_test[c] = feats_test[c].astype(pd.CategoricalDtype(categories=feats_train[c].cat.categories))

assert feats_train.isna().sum().sum() == 0 and feats_test.isna().sum().sum() == 0
print(f"{feats_train.shape[1]} features, {len(feats_train)} lignes train, {len(feats_test)} lignes test")

## 5. Modèles

Validation croisée stratifiée : il n'y a pas de colonne de date ni de chronologie
entre les lignes, la validation adversariale train/test donne une AUC de 0.50 et
le contrôle de fuite colonne par colonne ne trouve rien (AUC univarié max 0.57).
Les hyperparamètres sont ceux de `src/train.py` (`FINAL_*_PARAMS`).

**Modèles additifs.** Avec ces features, le score caché est une somme de fonctions
d'une seule feature : des arbres à une seule division (`max_depth=1` / `depth=1`)
font mieux que des arbres plus profonds, qui n'ajoutent que de la variance (LightGBM,
5 plis : profondeur 1 0.2310, 2 0.2316, 3 0.2324). Imposer le sens de variation des
facteurs dont la fonction de forme est monotone réduit encore le bruit.

**Lien probit.** La cible vaut 1 pour les 15 % de lignes dont « score caché + bruit »
est le plus élevé (exactement 1 500 sur 10 000 à chaque position du bloc client). Si
ce bruit est gaussien, P(y=1|x) = Φ(s(x) − seuil) : le score est additif sur l'échelle
probit, pas sur l'échelle logit. Entraîné avec l'objectif logit, le LightGBM additif
surestime les faibles risques (0.72 % prédit pour 0.39 % observé, écart de 3 σ) ;
entraîné avec la vraisemblance probit (objectif personnalisé ci-dessous), il est
calibré dans toutes les tranches de risque et gagne 0.0007 de logloss (5 plis).

**GAM probit.** Second modèle : les mêmes fonctions d'une feature, estimées par des
courbes lisses (P-splines) plutôt qu'en escalier, avec les mêmes contraintes de
monotonie. Code identique à `src/gam.py`.

In [ ]:
import numpy as np
from scipy import optimize, special
from scipy.interpolate import BSpline
from threadpoolctl import threadpool_limits

SPLINE_DEGREE = 3


def _rank_transform(train_sorted, x):
    """Rang moyen de x parmi les valeurs du train (ex aequo compris), ramené à [0, 1]."""
    below = np.searchsorted(train_sorted, x, side="left")
    below_or_equal = np.searchsorted(train_sorted, x, side="right")
    return (below + below_or_equal) / (2 * len(train_sorted))


class ProbitGAM:
    """Modèle additif probit à P-splines, avec contraintes de monotonie optionnelles.

    `monotone` : liste alignée sur les colonnes, 1 (croissante), -1 (décroissante) ou
    0 (libre).
    """

    def __init__(self, n_knots=20, lam=5.0, monotone=None, max_iter=5000):
        self.n_knots = n_knots
        self.lam = lam
        self.monotone = monotone
        self.max_iter = max_iter

    def _raw_basis(self, j, x):
        u = np.clip(_rank_transform(self.train_sorted_[j], x), 0.0, 1.0 - 1e-12)
        knots = np.r_[[0.0] * SPLINE_DEGREE, np.linspace(0, 1, self.n_knots + 2), [1.0] * SPLINE_DEGREE]
        return BSpline.design_matrix(u, knots, SPLINE_DEGREE).toarray()

    def _basis(self, j, x):
        return self._raw_basis(j, x) - self.basis_means_[j]

    def fit(self, X, y):
        with threadpool_limits(limits=1, user_api="blas"):
            return self._fit(X, y)

    def _fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        n, p = X.shape
        n_basis = self.n_knots + SPLINE_DEGREE + 1
        self.train_sorted_ = [np.sort(X[:, j]) for j in range(p)]
        self.basis_means_ = [self._raw_basis(j, X[:, j]).mean(axis=0) for j in range(p)]

        # Coefficients de la feature j : beta_j = [0, cumsum(delta_j)]. La courbe est
        # monotone si tous les incréments delta_j ont le même signe (bornes L-BFGS-B).
        # Comme la base est centrée, le premier coefficient est fixé à 0 sans perte.
        cumsum = np.vstack([np.zeros(n_basis - 1), np.tril(np.ones((n_basis - 1, n_basis - 1)))])
        second_diff = np.diff(np.eye(n_basis), 2, axis=0) @ cumsum
        penalty = second_diff.T @ second_diff
        design = np.hstack([self._basis(j, X[:, j]) @ cumsum for j in range(p)])
        k = n_basis - 1

        bounds = [(None, None)]
        for j in range(p):
            direction = 0 if self.monotone is None else self.monotone[j]
            bounds += [(0, None) if direction > 0 else (None, 0) if direction < 0 else (None, None)] * k

        def objective(theta):
            eta = theta[0] + design @ theta[1:]
            log_cdf, log_sf = special.log_ndtr(eta), special.log_ndtr(-eta)
            log_pdf = -0.5 * eta**2 - 0.5 * np.log(2 * np.pi)
            nll = -np.sum(np.where(y == 1, log_cdf, log_sf))
            grad_eta = np.where(y == 1, -np.exp(log_pdf - log_cdf), np.exp(log_pdf - log_sf))
            deltas = theta[1:].reshape(p, k)
            pen_grad = self.lam * deltas @ penalty
            value = nll + 0.5 * np.sum(pen_grad * deltas)
            grad = np.r_[grad_eta.sum(), design.T @ grad_eta + pen_grad.ravel()]
            return value / n, grad / n

        theta0 = np.r_[special.ndtri(y.mean()), np.zeros(p * k)]
        result = optimize.minimize(objective, theta0, jac=True, method="L-BFGS-B", bounds=bounds,
                                   options={"maxiter": self.max_iter, "maxfun": 2 * self.max_iter})
        self.intercept_ = result.x[0]
        self.coef_ = [cumsum @ d for d in result.x[1:].reshape(p, k)]
        self.n_iter_ = result.nit
        return self

    def decision_function(self, X):
        """Score η sur l'échelle probit."""
        X = np.asarray(X, dtype=float)
        eta = np.full(len(X), self.intercept_)
        with threadpool_limits(limits=1, user_api="blas"):
            for j, beta in enumerate(self.coef_):
                eta += self._basis(j, X[:, j]) @ beta
        return eta

    def predict_proba(self, X):
        p = special.ndtr(self.decision_function(X))
        return np.column_stack([1 - p, p])

In [ ]:
LGBM_PARAMS = dict(objective="binary", n_estimators=20000, learning_rate=0.05, max_depth=1, num_leaves=2,
                   min_child_samples=64, subsample=0.89, subsample_freq=1, colsample_bytree=0.5,
                   reg_lambda=0.013, verbosity=-1)
LGBM_ESR = 500
CATBOOST_PARAMS = dict(loss_function="Logloss", eval_metric="Logloss", iterations=20000, learning_rate=0.08,
                       depth=1, l2_leaf_reg=11.73, random_strength=0.0625, bagging_temperature=2.42, verbose=False)
CATBOOST_ESR = 500
GAM_KNOTS, GAM_LAMBDA = 20, 5.0
# Sens de variation imposés (1 : le risque augmente avec la feature, -1 : il diminue) : logique
# financière, confirmée par les fonctions de forme du modèle additif sans contrainte.
MONOTONE = {"outflow_over_inflow_rec": 1, "withdraw_over_inflow_rec": 1, "payments_value_ratio": 1,
            "daily_avg_bal_cv_6m": 1, "inflow_value_ratio": -1, "daily_avg_bal_momentum_recent_vs_old": -1,
            "bal_ratio": -1, "net_rec_over_inflow_old": -1, "x_90_d_activity_rate": -1}
LGBM_MAX_BIN = 1023


def zindi_score(ll, auc):
    return 0.4 * auc + 0.6 * (1 - ll / 0.595)


def folds_for(seed):
    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(feats_train, y))


def probit_grad_hess(eta, y_):
    # Gradient et hessienne, en eta, de la logloss -[y log Φ(eta) + (1-y) log(1-Φ(eta))], calculés en log.
    log_cdf, log_sf = special.log_ndtr(eta), special.log_ndtr(-eta)
    log_pdf = -0.5 * eta**2 - 0.5 * np.log(2 * np.pi)
    r1, r0 = np.exp(log_pdf - log_cdf), np.exp(log_pdf - log_sf)
    grad = np.where(y_ == 1, -r1, r0)
    hess = np.where(y_ == 1, r1 * (r1 + eta), r0 * (r0 - eta))
    return grad, hess


def probit_logloss(y_, eta):
    return -np.mean(np.where(y_ == 1, special.log_ndtr(eta), special.log_ndtr(-eta)))


def lgb_probit_objective(preds, data):
    # preds : score courant de LightGBM, qui inclut déjà l'init_score du Dataset.
    return probit_grad_hess(preds, data.get_label())


def lgb_probit_metric(preds, data):
    return "probit_logloss", probit_logloss(data.get_label(), preds), False


def run_lgbm(seed):
    Xtr, Xte = feats_train[selected], feats_test[selected]
    params = {k: v for k, v in LGBM_PARAMS.items() if k not in ("objective", "n_estimators")}
    params.update(max_bin=LGBM_MAX_BIN, monotone_constraints=[MONOTONE.get(c, 0) for c in selected],
                  random_state=seed, objective=lgb_probit_objective, metric="None")
    oof, test_pred = np.zeros(len(y)), np.zeros(len(feats_test))
    for tri, vai in folds_for(seed):
        init = special.ndtri(y[tri].mean())  # le score part de Φ⁻¹(taux de positifs)
        dtr = lgb.Dataset(Xtr.iloc[tri], y[tri].astype(float), init_score=np.full(len(tri), init),
                          categorical_feature=selected_cats, params={"max_bin": LGBM_MAX_BIN})
        dva = lgb.Dataset(Xtr.iloc[vai], y[vai].astype(float), init_score=np.full(len(vai), init), reference=dtr)
        bst = lgb.train(params, dtr, num_boost_round=LGBM_PARAMS["n_estimators"], valid_sets=[dva],
                        feval=lgb_probit_metric, callbacks=[lgb.early_stopping(LGBM_ESR, verbose=False)])
        oof[vai] = special.ndtr(bst.predict(Xtr.iloc[vai], num_iteration=bst.best_iteration) + init)
        test_pred += special.ndtr(bst.predict(Xte, num_iteration=bst.best_iteration) + init) / N_FOLDS
    return oof, test_pred


def run_gam(seed):
    Xtr, Xte = feats_train[selected], feats_test[selected]
    mono = [MONOTONE.get(c, 0) for c in selected]
    oof, test_pred = np.zeros(len(y)), np.zeros(len(feats_test))
    for tri, vai in folds_for(seed):
        m = ProbitGAM(n_knots=GAM_KNOTS, lam=GAM_LAMBDA, monotone=mono).fit(Xtr.iloc[tri], y[tri])
        oof[vai] = m.predict_proba(Xtr.iloc[vai])[:, 1]
        test_pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, test_pred


def rank_features():
    # Gain cumulé du LightGBM additif (objectif logit) sur 5 plis (seed fixe) : même classement que
    # src/train.py. Le même classement fait avec l'objectif probit retient les mêmes 15 features.
    gain = np.zeros(feats_train.shape[1])
    for tri, vai in StratifiedKFold(5, shuffle=True, random_state=SEED).split(feats_train, y):
        m = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=SEED)
        m.fit(feats_train.iloc[tri], y[tri], eval_set=[(feats_train.iloc[vai], y[vai])],
              categorical_feature=cat_cols, callbacks=[lgb.early_stopping(LGBM_ESR, verbose=False)])
        gain += m.booster_.feature_importance("gain")
    return pd.Series(gain, index=feats_train.columns).sort_values(ascending=False)


def run_catboost(seed):
    Xtr, Xte = feats_train[selected], feats_test[selected]
    oof, test_pred = np.zeros(len(y)), np.zeros(len(feats_test))
    cat_idx = [Xtr.columns.get_loc(c) for c in selected_cats]
    for tri, vai in folds_for(seed):
        m = cb.CatBoostClassifier(**CATBOOST_PARAMS, random_seed=seed)
        m.fit(Xtr.iloc[tri], y[tri], eval_set=(Xtr.iloc[vai], y[vai]), cat_features=cat_idx,
              early_stopping_rounds=CATBOOST_ESR, verbose=False)
        oof[vai] = m.predict_proba(Xtr.iloc[vai])[:, 1]
        test_pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, test_pred


def bagged(run_fn, name):
    runs = [run_fn(s) for s in BAGGING_SEEDS]
    oof = np.mean([r[0] for r in runs], axis=0)
    test_pred = np.mean([r[1] for r in runs], axis=0)
    ll, auc = log_loss(y, oof), roc_auc_score(y, oof)
    print(f"{name:16s} logloss={ll:.5f} auc={auc:.5f} score estimé={zindi_score(ll, auc):.5f}")
    return oof, test_pred

**Sélection de features.** Les 15 features les mieux classées portent tout le
signal : au-delà, on ajoute de la variance (sélection refaite dans chaque pli, sans
biais : 0.2298 avec 15 features contre 0.2305 avec les 137, voir experiments.md).

In [ ]:
N_FEATURES = 15
ranking = rank_features()
selected = list(ranking.index[:N_FEATURES])
selected_cats = [c for c in cat_cols if c in selected]
print(f"{len(selected)} features sélectionnées :", ", ".join(selected))

In [ ]:
oof_lgbm, test_lgbm = bagged(run_lgbm, "LightGBM probit")
oof_gam, test_gam = bagged(run_gam, "GAM probit")
oof_cb, test_cb = bagged(run_catboost, "CatBoost")

## 6. Blend sur l'échelle probit

Les trois modèles sont moyennés sur l'échelle probit (moyenne pondérée de Φ⁻¹(p)) :
une somme pondérée de modèles additifs reste additive. Les poids (grille au pas de
0.05, de somme 1) maximisent le score Zindi sur les prédictions OOF. Pas de
recalibration : le blend est calibré dans toutes les tranches de risque (tableau
ci-dessous), et une recalibration de Platt ajustée en validation croisée ne gagne rien.

In [ ]:
oof_all = {"lightgbm": oof_lgbm, "gam": oof_gam, "catboost": oof_cb}
test_all = {"lightgbm": test_lgbm, "gam": test_gam, "catboost": test_cb}


def probit_blend(preds, weights):
    eta = sum(w * special.ndtri(np.clip(preds[name], 1e-15, 1 - 1e-15)) for name, w in weights.items())
    return special.ndtr(eta)


grid = np.round(np.arange(0.0, 1.0 + 1e-9, 0.05), 10)
best_score, weights = -np.inf, None
for w_lgbm, w_gam in itertools.product(grid, repeat=2):
    if w_lgbm + w_gam > 1.0 + 1e-9:
        continue
    candidate = {"lightgbm": w_lgbm, "gam": w_gam, "catboost": max(0.0, round(1.0 - w_lgbm - w_gam, 10))}
    p = probit_blend(oof_all, candidate)
    score = zindi_score(log_loss(y, p), roc_auc_score(y, p))
    if score > best_score:
        best_score, weights = score, candidate

oof_blend = probit_blend(oof_all, weights)
test_pred = probit_blend(test_all, weights)
ll, auc = log_loss(y, oof_blend), roc_auc_score(y, oof_blend)
print("poids :", {k: float(v) for k, v in weights.items()})
print(f"blend probit logloss={ll:.5f} auc={auc:.5f} score estimé={zindi_score(ll, auc):.5f}")

calib = pd.DataFrame({"y": y, "p": oof_blend})
calib = calib.groupby(pd.cut(calib.p, [0, .005, .01, .02, .05, .1, .2, .4, .6, .8, .9, .95, .99, 1]), observed=True).agg(
    n=("y", "size"), predit=("p", "mean"), observe=("y", "mean"))
calib["ecart_en_sigma"] = (calib.observe - calib.predit) / np.sqrt(calib.predit * (1 - calib.predit) / calib.n)
calib.round(4)

## 7. Soumission

Colonnes `ID, Target`, avec des probabilités brutes, jamais arrondies ni seuillées
(règle du défi).

In [ ]:
submission = pd.DataFrame({"ID": feats_test.index, "Target": test_pred})
assert submission.shape == (len(test), 2) and submission["Target"].between(0, 1).all()
submission.to_csv("submission.csv", index=False)
print(submission.describe())
try:
    from google.colab import files
    files.download("submission.csv")
except ImportError:
    print("Fichier sauvegardé : submission.csv")